In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

# Cargar dataset
df = pd.read_csv("prosperLoanData1.csv")


#Estados de la variable objetivo
status_map = {
    "Completed": 1,
    "FinalPaymentInProgress": 1,
    "Defaulted": 0,
    "Chargedoff": 0,
    "Past Due": None,
    "Current": None,
    "Cancelled": None
}

#definición de la variable objetivo
df["recuperado"] = df["LoanStatus"].map(status_map)
df_modelo = df.dropna(subset=["recuperado"])
##print (df_modelo["recuperado"].value_counts())

##Variables predictoras escogidas
features = ["BorrowerRate", "LoanOriginalAmount", "IncomeRange", "Term"]

##Matriz de características y variable objetivo
X = df_modelo[features]
y = df_modelo["recuperado"]

##división de los datos en conjunto de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split( 
    X, y, test_size=0.2, random_state=42, stratify=y #random_state=42 asegura que la división sea siempre igual (reproducible) y test size 20%
)

##Tipo de datos de las variables predictoras
num_features = ["BorrowerRate", "LoanOriginalAmount", "Term"]
cat_features = ["IncomeRange"]

##Escalado de las variables numéricas y codificación one-hot de las variables categóricas
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features), ##standardscaler media 0 y desviación estándar 1
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)
    ]
)


